# Setup FastAPI + Ngrok trên Kaggle
Chạy từng cell theo thứ tự

In [1]:
# Cell 1: Cài dependencies
!pip install -q fastapi uvicorn[standard] qdrant-client pymongo \
             transformers torch Pillow numpy python-dotenv \
             deep-translator google-genai pydantic pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.1 MB/s eta 0:00:00


In [ ]:
# Cell 2: Clone hoặc upload project
# Nếu project trên GitHub:
!git clone https://github.com/triet-bit/video_search.git
%cd video_search

# Nếu upload zip lên Kaggle dataset thì dùng:
# import shutil
# shutil.unpack_archive('/kaggle/input/<dataset-name>/renovateHCMAI.zip', '.')
# %cd video_search

/bin/bash: line 1: your-username: No such file or directory
[Errno 2] No such file or directory: 'renovateHCMAI'
/kaggle/working


# Bước 3: Cấu hình .env (Cloud DB)
Do Kaggle không hỗ trợ chạy Docker demon, chúng ta phải dùng Cloud DB (MongoDB Atlas & Qdrant Cloud).
Tạo một file `.env` trong thư mục `video_search` (hãy tạo file này ở tab bên phải -> `files` hoặc chạy lệnh bên dưới với thông tin thật của bạn):

In [ ]:
%%writefile .env
MONGO_URI="mongodb+srv://<user>:<password>@cluster/"
MONGO_DB="HCMAIDB"
QDRANT_URL="https://<your-qdrant-url>.cloud.qdrant.io"
QDRANT_API_KEY="<your-qdrant-api-key>"
ENABLE_RERANKER=false
GOOGLE_API_KEY="<your-google-api-key>"


In [5]:
# Cell 5: Setup ngrok auth token
# Lấy token tại: https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "your_ngrok_auth_token_here"  # <-- thay vào đây
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [6]:
# Cell 6: Chạy FastAPI + expose qua ngrok
import subprocess
import time
import sys
import os
from pyngrok import ngrok

# Thêm project root vào sys.path
sys.path.insert(0, os.getcwd())

# Chạy uvicorn trong background
server = subprocess.Popen(
    ["uvicorn", "src.api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(5)  # Đợi server khởi động

# Tạo tunnel ngrok
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print(f"✓ Server running")
print(f"✓ Public URL : {public_url}")
print(f"✓ Swagger UI : {public_url}/docs")
print(f"✓ ReDoc      : {public_url}/redoc")

t=2026-03-22T12:11:12+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: your_ngrok_auth_token_here\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:  authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.
ERROR:  Your authtoken: your_ngrok_auth_token_here
ERROR:  Instructions to install your authtoken are on your ngrok dashboard:
ERROR:  https://dashboard.ngrok.com/get-started/your-authtoken
ERROR:  
ERROR:  ERR_NGROK_105
ERROR:  https://ngrok.com/docs/errors/err_ngrok_105
ERROR:  
t=2026-03-22T12:11:12+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: your_ngrok_auth_token_here\nInst

PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: your_ngrok_auth_token_here\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.

In [ ]:
# Cell 7: Xem log server nếu có lỗi
stdout, stderr = server.communicate(timeout=3)
print(stderr.decode())

In [ ]:
# Cell 8: Test health check
import requests

resp = requests.get(f"{public_url}/health")
print(resp.json())

In [ ]:
# Cell 9: Dừng server và ngrok khi xong
ngrok.disconnect(tunnel.public_url)
ngrok.kill()
server.terminate()
print('✓ Server stopped')